# 3. Model sensitivity analysis (CH4 util-free)

**Purpose:** CTS-2026-0235R2 Associate Editor supplement — refit the CH4 (`non_opioid_ed`) model **without utilization-derived features** and compare holdout AUPRC, top drug SHAP ranks, and published pair/triplet persistence vs the primary framing.

**Template:** Same EC2 root-notebook style as [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb) (setup → sync → scoped run → artifacts on disk/S3). This notebook does **not** retrain the full Step 6/7/8 production stack.

**Production runner:** `6_final_model/run_sensitivity_util_free.py` (SSOT). Do not use abandoned `ch04_util*` intermediates under `notebooks/dev/`.

**Default scope:** `non_opioid_ed` / `65-74` aggregate util-free refit (XGBoost). Expand config cell only if needed.

**Outputs (not embedded):**
- `6_final_model/outputs/non_opioid_ed/65_74_util_free/`
- `8_ffa_analysis/outputs/non_opioid_ed/65_74_util_free_sensitivity/`
- `manuscript/data/supplementary/ch04_util_free_sensitivity/`

**Docs:** `manuscript/data/supplementary/ch04_util_free_sensitivity/README.md` · lessons learned (final production workflow).

Run from **repo root** on EC2 (NVMe data root via `get_data_root()`).


In [ ]:
# Setup: paths and project root (match notebook 3 template)
import os
import sys
import subprocess
import shutil
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "10_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "10_risk_dashboard").exists():
    pass
else:
    for cand in (Path.cwd(), *Path.cwd().parents):
        if (cand / "6_final_model").exists() and (cand / "py_helpers").exists():
            PROJECT_ROOT = cand
            break

sys.path.insert(0, str(PROJECT_ROOT))
from py_helpers.env_utils import get_data_root, get_model_data_root

S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
MODEL_DATA_ROOT = get_model_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
SENSITIVITY_SCRIPT = PROJECT_ROOT / "6_final_model" / "run_sensitivity_util_free.py"

print("PGx model sensitivity workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Model data root: {MODEL_DATA_ROOT}")
print(f"Final model outputs: {FINAL_MODEL_OUTPUTS}")
print(f"Sensitivity script: {SENSITIVITY_SCRIPT}")
print("=" * 60)
assert SENSITIVITY_SCRIPT.exists(), f"Missing runner: {SENSITIVITY_SCRIPT}"


## Config

Default matches the CH4 R2 util-free supplement (manuscript SSOT). Edit only when expanding bands intentionally.


In [ ]:
# Sensitivity run scope (CH4 util-free SSOT)
COHORT = "non_opioid_ed"
AGE_BAND = "65-74"
AGE_FNAME = "65_74"

# Optional: sync production Step-6 / model_events inputs before refit
SYNC_FINAL_MODEL_FROM_S3 = True
SYNC_MODEL_EVENTS_FROM_S3 = False  # set True on EC2 if local model_events missing

MS_DIR = PROJECT_ROOT / "manuscript" / "data" / "supplementary" / "ch04_util_free_sensitivity"
COMP_DIR = (
    PROJECT_ROOT
    / "8_ffa_analysis"
    / "outputs"
    / COHORT
    / f"{AGE_FNAME}_util_free_sensitivity"
)
SENS_DIR = FINAL_MODEL_OUTPUTS / COHORT / f"{AGE_FNAME}_util_free"

print(f"Cohort / age band: {COHORT} / {AGE_BAND}")
print(f"Sensitivity model dir: {SENS_DIR}")
print(f"Comparison dir: {COMP_DIR}")
print(f"Manuscript supplementary: {MS_DIR}")


## Sync required inputs from S3 (idempotent)

Sync **Step 6 final-model** gold artifacts for the configured cohort/age band so the util-free refit can use the same holdout feature tables / metrics when present. Uses `aws s3 sync` (profile from `AWS_PROFILE` when set).


In [ ]:
# Sync final_model gold prefix for this cohort/age band (idempotent)
_aws = shutil.which("aws") or "aws"
_profile = ["--profile", AWS_PROFILE] if AWS_PROFILE else []

if SYNC_FINAL_MODEL_FROM_S3:
    prefixes = [
        f"gold/final_model/{COHORT}/{AGE_BAND}",
        f"gold/manuscript/final_model/{COHORT}/{AGE_BAND}",
    ]
    local_dest = FINAL_MODEL_OUTPUTS / COHORT / AGE_FNAME
    local_dest.mkdir(parents=True, exist_ok=True)
    for prefix in prefixes:
        uri = f"s3://{S3_BUCKET}/{prefix}"
        print(f"Sync {uri} -> {local_dest}")
        r = subprocess.run(
            [_aws, "s3", "sync", uri, str(local_dest)] + _profile,
            capture_output=True,
            text=True,
        )
        if r.returncode == 0:
            print(f"  OK ({prefix})")
        else:
            print(f"  skip/warn exit={r.returncode}: {(r.stderr or r.stdout or '')[:400]}")
else:
    print("SYNC_FINAL_MODEL_FROM_S3=False; using local artifacts only")

if SYNC_MODEL_EVENTS_FROM_S3:
    me_prefix = f"gold/cohorts_model_data/cohort_name={COHORT}/age_band={AGE_BAND}"
    uri = f"s3://{S3_BUCKET}/{me_prefix}"
    dest = Path(MODEL_DATA_ROOT)
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Sync {uri} -> {dest}")
    r = subprocess.run(
        [_aws, "s3", "sync", uri, str(dest)] + _profile,
        capture_output=True,
        text=True,
    )
    print("  OK" if r.returncode == 0 else f"  warn exit={r.returncode}")


## Run utilization-free sensitivity (CH4)

Executes `6_final_model/run_sensitivity_util_free.py` in-process from repo root. Writes comparison CSVs + `sensitivity_summary.json` to disk/manuscript supplementary — prints a short summary only (no large embeds).


In [ ]:
# Run production sensitivity runner
import runpy

os.chdir(PROJECT_ROOT)
print(f"Executing {SENSITIVITY_SCRIPT.relative_to(PROJECT_ROOT)} ...")
runpy.run_path(str(SENSITIVITY_SCRIPT), run_name="__main__")
print("Sensitivity runner finished.")


## Load summary paths (machine-readable SSOT)

Prefer reading `sensitivity_summary.json` from manuscript supplementary for response-letter / supplement prose. Do not paste full SHAP tables into the notebook.


In [ ]:
# Print headline summary only
import json

summary_path = MS_DIR / "sensitivity_summary.json"
if not summary_path.exists():
    summary_path = COMP_DIR / "sensitivity_summary.json"

assert summary_path.exists(), f"Missing summary: {summary_path}"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

keys = [
    "cohort",
    "age_band",
    "n_util_features_dropped",
    "util_free_auprc",
    "util_free_pr_lift",
    "manuscript_primary_auprc_65_74",
    "auprc_delta_vs_manuscript",
    "published_pairs_positive_ie",
]
print("Summary file:", summary_path)
for k in keys:
    print(f"  {k}: {summary.get(k)}")
overlap = summary.get("drug_shap_overlap") or {}
print(f"  top_drug_jaccard: {overlap.get('jaccard')}")
print(f"  intersection_n: {overlap.get('n_intersection')}")
print("\nArtifacts:")
print(" ", SENS_DIR)
print(" ", COMP_DIR)
print(" ", MS_DIR)
print("\nResponse blurb length:", len(summary.get("response_letter_blurb") or ""), "chars")
